# 🧪 Interactive LIME Explainer — Launcher

This notebook launches an **interactive Streamlit app** that lets you:
1. Type or paste any text
2. Classify it as Fake/Real using **BERT** and/or **TPA-BERT**
3. See **LIME explanations** with highlighted text and feature importance charts
4. **Compare both models side-by-side**

**Instructions:**
1. Set runtime to **TPU** (Runtime → Change runtime type → TPU)
2. Run all cells in order
3. Click the **ngrok URL** printed in Cell 3 to open the app

## Cell 1 — Mount Drive & Install Dependencies

In [1]:
# Mount Google Drive (models are stored here)
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install -q streamlit lime pyngrok transformers torch

# Verify model paths exist
import os

bert_path = "/content/drive/MyDrive/bert_models/WELFake"
tpa_path = "/content/drive/MyDrive/models/AdversarialBERT_WELFake/adversarial_bert.pt"

print(f"\n{'='*60}")
print(f"BERT model:     {'✓ Found' if os.path.isdir(bert_path) else '✗ NOT FOUND'}")
print(f"TPA-BERT model: {'✓ Found' if os.path.isfile(tpa_path) else '✗ NOT FOUND'}")
print(f"{'='*60}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 11.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 158.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 188.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 6.1 MB/s eta 0:00:00
error: uninstall-distutils-installed-package

× Cannot uninstall blinker 1.4
╰─> It is a distutils installed project and thus we cannot accurately determine which files belong to it which would lead to only a partial uninstall.

BERT model:     ✓ Found
TPA-BERT model: 

## Cell 2 — Write Streamlit App to Disk

This cell writes the full Streamlit application to `/content/app.py`.
Edit the `BERT_MODEL_PATH` or `TPA_BERT_MODEL_DIR` variables at the top if your paths differ.

In [2]:
%%writefile /content/app.py
"""
Interactive LIME Explainability App
────────────────────────────────────
Streamlit app that loads BERT and/or TPA-BERT (AdversarialBERT) models,
classifies user-typed text as Fake or Real news, and generates live
LIME explanations.

Designed to run on Google Colab via ngrok tunnel.
"""

import streamlit as st
import torch
import torch.nn as nn
import numpy as np
import re
import os
import time
from transformers import BertTokenizer, BertModel, BertForSequenceClassification
from lime.lime_text import LimeTextExplainer

# ── Configuration ────────────────────────────────────────────
BERT_MODEL_PATH = "/content/drive/MyDrive/bert_models/WELFake"
TPA_BERT_MODEL_DIR = "/content/drive/MyDrive/models/AdversarialBERT_WELFake"
TPA_BERT_WEIGHTS = "adversarial_bert.pt"

CLASS_NAMES = ["Fake", "Real"]
DEVICE = torch.device("cpu")  # CPU for LIME compat (many small fwd passes)


# ── Model Definitions ───────────────────────────────────────
class AdversarialBERT(nn.Module):
    """TPA-BERT model architecture (must match saved weights)."""

    def __init__(self, num_labels=2, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        if token_type_ids is not None:
            out = self.bert(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        else:
            out = self.bert(
                input_ids=input_ids, attention_mask=attention_mask
            )
        cls_emb = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_emb))
        return logits, cls_emb


# ── Model Loading (cached) ──────────────────────────────────
@st.cache_resource
def load_bert_model():
    """Load standard BERT model + tokenizer from Google Drive."""
    model = BertForSequenceClassification.from_pretrained(BERT_MODEL_PATH)
    tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_PATH)
    model.to(DEVICE)
    model.eval()
    param_count = sum(p.numel() for p in model.parameters())
    return model, tokenizer, param_count


@st.cache_resource
def load_tpa_bert_model():
    """Load TPA-BERT (AdversarialBERT) model + tokenizer from Google Drive."""
    model = AdversarialBERT()
    weights_path = os.path.join(TPA_BERT_MODEL_DIR, TPA_BERT_WEIGHTS)
    state_dict = torch.load(weights_path, map_location="cpu")
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    tokenizer = BertTokenizer.from_pretrained(TPA_BERT_MODEL_DIR)
    param_count = sum(p.numel() for p in model.parameters())
    return model, tokenizer, param_count


# ── Prediction Functions ────────────────────────────────────
def make_predict_fn(model, tokenizer, is_tpa=False):
    """Create a LIME-compatible prediction function.

    Returns: texts (list[str]) → np.array of shape (N, 2)
    where columns are [P(Fake), P(Real)].
    """

    def predict_proba(texts):
        all_probs = []
        batch_size = 64
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            encodings = tokenizer(
                batch,
                truncation=True,
                padding=True,
                max_length=128,
                return_tensors="pt",
            )
            encodings = {k: v.to(DEVICE) for k, v in encodings.items()}

            with torch.no_grad():
                if is_tpa:
                    outputs = model(**encodings)
                    logits = (
                        outputs[0]
                        if isinstance(outputs, (tuple, list))
                        else outputs.logits
                    )
                else:
                    outputs = model(**encodings)
                    logits = outputs.logits

                probs = torch.nn.functional.softmax(logits, dim=-1)
                all_probs.append(probs.cpu().numpy())

        return np.concatenate(all_probs, axis=0)

    return predict_proba


# ── Page Config ──────────────────────────────────────────────
st.set_page_config(
    page_title="LIME Interactive Explainer",
    page_icon="🧪",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ── Custom CSS ───────────────────────────────────────────────
st.markdown(
    """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

html, body, [class*="st-"] {
    font-family: 'Inter', sans-serif;
}

header[data-testid="stHeader"] {
    background: rgba(15, 15, 26, 0.95) !important;
    backdrop-filter: blur(8px);
    -webkit-backdrop-filter: blur(8px);
}

.stApp {
    background: linear-gradient(160deg, #0b0b15 0%, #131325 40%, #162035 100%);
}

section[data-testid="stSidebar"] {
    background: rgba(12, 12, 22, 0.97);
    border-right: 1px solid rgba(255, 255, 255, 0.06);
}
section[data-testid="stSidebar"] .stMarkdown hr {
    border-color: rgba(255, 255, 255, 0.06);
}

div[data-testid="stMetric"] {
    background: rgba(255, 255, 255, 0.04);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 12px;
    padding: 16px 20px;
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
    transition: transform 0.2s ease, border-color 0.2s ease;
}
div[data-testid="stMetric"]:hover {
    transform: translateY(-2px);
    border-color: rgba(255, 255, 255, 0.15);
}
div[data-testid="stMetric"] label {
    color: rgba(255, 255, 255, 0.55) !important;
    font-weight: 500;
    letter-spacing: 0.03em;
    text-transform: uppercase;
    font-size: 0.7rem !important;
}
div[data-testid="stMetric"] [data-testid="stMetricValue"] {
    font-weight: 700;
    font-size: 1.6rem !important;
}

.glass-card {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 16px;
    padding: 28px 32px;
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
    margin: 8px 0;
    line-height: 1.85;
    font-size: 0.95rem;
    color: rgba(255, 255, 255, 0.8);
    max-height: 400px;
    overflow-y: auto;
    word-wrap: break-word;
}

.lime-word-pos {
    border-radius: 4px;
    padding: 2px 5px;
    margin: 0 1px;
    cursor: default;
    color: #fff;
    display: inline;
    text-decoration: none;
}
.lime-word-neg {
    border-radius: 4px;
    padding: 2px 5px;
    margin: 0 1px;
    cursor: default;
    color: #fff;
    display: inline;
    text-decoration: none;
}

.bar-chart-container {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 16px;
    padding: 24px 28px;
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
}
.bar-row {
    display: flex;
    align-items: center;
    margin: 6px 0;
    gap: 12px;
}
.bar-word {
    width: 140px;
    text-align: right;
    font-size: 0.85rem;
    font-weight: 500;
    color: rgba(255, 255, 255, 0.8);
    flex-shrink: 0;
    font-family: 'Inter', monospace;
}
.bar-track {
    flex: 1;
    height: 22px;
    background: rgba(255, 255, 255, 0.04);
    border-radius: 6px;
    position: relative;
    overflow: hidden;
}
.bar-fill-pos {
    position: absolute;
    left: 50%;
    height: 100%;
    background: linear-gradient(90deg, rgba(0, 212, 170, 0.7), rgba(0, 212, 170, 0.95));
    border-radius: 0 6px 6px 0;
    transition: width 0.5s ease;
}
.bar-fill-neg {
    position: absolute;
    right: 50%;
    height: 100%;
    background: linear-gradient(270deg, rgba(255, 75, 110, 0.7), rgba(255, 75, 110, 0.95));
    border-radius: 6px 0 0 6px;
    transition: width 0.5s ease;
}
.bar-center-line {
    position: absolute;
    left: 50%;
    top: 0;
    bottom: 0;
    width: 1px;
    background: rgba(255, 255, 255, 0.15);
}
.bar-value {
    width: 70px;
    font-size: 0.78rem;
    font-weight: 600;
    color: rgba(255, 255, 255, 0.6);
    flex-shrink: 0;
    font-family: 'Inter', monospace;
}

.section-header {
    font-size: 0.75rem;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: rgba(255, 255, 255, 0.4);
    margin: 24px 0 12px 0;
    padding-bottom: 8px;
    border-bottom: 1px solid rgba(255, 255, 255, 0.06);
}

.label-badge {
    display: inline-block;
    padding: 3px 10px;
    border-radius: 6px;
    font-size: 0.78rem;
    font-weight: 600;
    letter-spacing: 0.03em;
}
.label-real {
    background: rgba(0, 212, 170, 0.15);
    color: #00d4aa;
    border: 1px solid rgba(0, 212, 170, 0.3);
}
.label-fake {
    background: rgba(255, 75, 110, 0.15);
    color: #ff4b6e;
    border: 1px solid rgba(255, 75, 110, 0.3);
}

.legend {
    display: flex;
    gap: 20px;
    margin: 8px 0 4px 0;
    font-size: 0.78rem;
    color: rgba(255, 255, 255, 0.5);
}
.legend-dot {
    display: inline-block;
    width: 10px;
    height: 10px;
    border-radius: 50%;
    margin-right: 6px;
    vertical-align: middle;
}

.main-title {
    font-size: 1.8rem;
    font-weight: 700;
    background: linear-gradient(135deg, #00d4aa 0%, #7b68ee 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 4px;
}
.main-subtitle {
    font-size: 0.9rem;
    color: rgba(255, 255, 255, 0.4);
    margin-bottom: 24px;
}

.confidence-bar-outer {
    width: 100%;
    height: 28px;
    background: rgba(255, 255, 255, 0.04);
    border-radius: 8px;
    position: relative;
    overflow: hidden;
    margin: 8px 0;
    border: 1px solid rgba(255, 255, 255, 0.08);
}
.confidence-bar-fake {
    position: absolute;
    left: 0;
    top: 0;
    height: 100%;
    background: linear-gradient(90deg, rgba(255, 75, 110, 0.8), rgba(255, 75, 110, 0.5));
    transition: width 0.6s ease;
}
.confidence-bar-real {
    position: absolute;
    right: 0;
    top: 0;
    height: 100%;
    background: linear-gradient(270deg, rgba(0, 212, 170, 0.8), rgba(0, 212, 170, 0.5));
    transition: width 0.6s ease;
}
.confidence-labels {
    display: flex;
    justify-content: space-between;
    font-size: 0.75rem;
    color: rgba(255, 255, 255, 0.5);
    margin-top: 4px;
}

.model-header {
    background: rgba(255, 255, 255, 0.04);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 12px;
    padding: 14px 20px;
    margin-bottom: 16px;
    text-align: center;
}
.model-header h3 {
    margin: 0;
    font-size: 1.1rem;
    font-weight: 600;
    color: rgba(255, 255, 255, 0.85);
}
.model-header .model-meta {
    font-size: 0.72rem;
    color: rgba(255, 255, 255, 0.35);
    margin-top: 4px;
}

.status-loaded {
    color: #00d4aa;
    font-size: 0.8rem;
}
.status-missing {
    color: #ff4b6e;
    font-size: 0.8rem;
}

.glass-card::-webkit-scrollbar { width: 6px; }
.glass-card::-webkit-scrollbar-track { background: transparent; }
.glass-card::-webkit-scrollbar-thumb {
    background: rgba(255,255,255,0.15);
    border-radius: 3px;
}

div[data-testid="stTextArea"] textarea {
    background: rgba(255, 255, 255, 0.04) !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 12px !important;
    color: rgba(255, 255, 255, 0.85) !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 0.95rem !important;
    line-height: 1.7 !important;
}
div[data-testid="stTextArea"] textarea:focus {
    border-color: rgba(123, 104, 238, 0.5) !important;
    box-shadow: 0 0 0 1px rgba(123, 104, 238, 0.3) !important;
}
</style>
""",
    unsafe_allow_html=True,
)


# ── Helper Functions ─────────────────────────────────────────
def label_badge(label_idx: int) -> str:
    """Return an HTML badge for a label."""
    name = CLASS_NAMES[label_idx]
    css_class = "label-real" if name == "Real" else "label-fake"
    return f'<span class="label-badge {css_class}">{name}</span>'


def build_highlighted_text(text: str, lime_weights: list) -> str:
    """Build HTML with LIME-highlighted words."""
    if not lime_weights:
        return f'<div class="glass-card">{text}</div>'

    weight_map = {}
    for word, weight in lime_weights:
        weight_map[word.lower()] = weight

    max_abs = max(abs(w) for _, w in lime_weights) if lime_weights else 1.0
    if max_abs == 0:
        max_abs = 1.0

    tokens = text.split()
    html_parts = []
    for token in tokens:
        clean = re.sub(r"[^\w]", "", token).lower()
        if clean in weight_map:
            w = weight_map[clean]
            intensity = min(abs(w) / max_abs, 1.0) * 0.6 + 0.15
            if w > 0:
                bg = f"rgba(0, 212, 170, {intensity:.2f})"
                border_c = f"rgba(0, 212, 170, {min(intensity + 0.2, 1.0):.2f})"
            else:
                bg = f"rgba(255, 75, 110, {intensity:.2f})"
                border_c = f"rgba(255, 75, 110, {min(intensity + 0.2, 1.0):.2f})"
            sign = "+" if w > 0 else ""
            html_parts.append(
                f'<span class="lime-word-{"pos" if w > 0 else "neg"}" '
                f'style="background:{bg}; border-bottom: 2px solid {border_c};" '
                f'title="{sign}{w:.4f}">{token}</span>'
            )
        else:
            html_parts.append(token)

    highlighted = " ".join(html_parts)
    return f'<div class="glass-card">{highlighted}</div>'


def build_bar_chart(lime_weights: list, n_features: int = 15) -> str:
    """Build a custom HTML horizontal bar chart for LIME feature weights."""
    if not lime_weights:
        return ""

    sorted_weights = sorted(lime_weights, key=lambda x: abs(x[1]), reverse=True)[
        :n_features
    ]
    max_abs = max(abs(w) for _, w in sorted_weights) if sorted_weights else 1.0
    if max_abs == 0:
        max_abs = 1.0

    rows = []
    for word, weight in sorted_weights:
        pct = (abs(weight) / max_abs) * 48
        if weight >= 0:
            fill = f'<div class="bar-fill-pos" style="width: {pct}%;"></div>'
        else:
            fill = f'<div class="bar-fill-neg" style="width: {pct}%;"></div>'

        sign = "+" if weight >= 0 else ""
        rows.append(
            f'<div class="bar-row">'
            f'  <div class="bar-word">{word}</div>'
            f'  <div class="bar-track">{fill}<div class="bar-center-line"></div></div>'
            f'  <div class="bar-value">{sign}{weight:.4f}</div>'
            f"</div>"
        )

    return '<div class="bar-chart-container">' + "\n".join(rows) + "</div>"


def build_confidence_bar(probs: list) -> str:
    """Build a horizontal stacked bar showing P(Fake) vs P(Real)."""
    p_fake = probs[0] * 100
    p_real = probs[1] * 100
    return (
        f'<div class="confidence-bar-outer">'
        f'  <div class="confidence-bar-fake" style="width: {p_fake}%;"></div>'
        f'  <div class="confidence-bar-real" style="width: {p_real}%;"></div>'
        f'</div>'
        f'<div class="confidence-labels">'
        f'  <span>Fake: {p_fake:.1f}%</span>'
        f'  <span>Real: {p_real:.1f}%</span>'
        f'</div>'
    )


def render_explanation(
    model_name: str,
    param_count: int,
    pred_label: int,
    probs: list,
    lime_weights: list,
    text: str,
    n_chart_features: int,
    elapsed: float,
):
    """Render a single model's explanation results."""
    st.markdown(
        f'<div class="model-header">'
        f"<h3>{model_name}</h3>"
        f'<div class="model-meta">{param_count:,} parameters</div>'
        f"</div>",
        unsafe_allow_html=True,
    )

    st.markdown(
        f"**Prediction:** {label_badge(pred_label)} &nbsp; "
        f"(**{probs[pred_label]*100:.1f}%** confidence)",
        unsafe_allow_html=True,
    )

    st.markdown(build_confidence_bar(probs), unsafe_allow_html=True)
    st.caption(f"⏱ Explanation generated in {elapsed:.1f}s")

    st.markdown(
        '<div class="section-header">Highlighted Text</div>',
        unsafe_allow_html=True,
    )
    st.markdown(
        f'<div class="legend">'
        f'<span><span class="legend-dot" style="background:#00d4aa;"></span>'
        f"Supports prediction</span>"
        f'<span><span class="legend-dot" style="background:#ff4b6e;"></span>'
        f"Contradicts prediction</span>"
        f"</div>",
        unsafe_allow_html=True,
    )
    st.markdown(
        build_highlighted_text(text, lime_weights), unsafe_allow_html=True
    )

    st.markdown(
        f'<div class="section-header">Feature Importance — Top {n_chart_features}</div>',
        unsafe_allow_html=True,
    )
    st.markdown(
        build_bar_chart(lime_weights, n_features=n_chart_features),
        unsafe_allow_html=True,
    )


# ── Main App ─────────────────────────────────────────────────
def main():
    st.markdown(
        '<div class="main-title">🧪 Interactive LIME Explainer</div>',
        unsafe_allow_html=True,
    )
    st.markdown(
        '<div class="main-subtitle">'
        "Type or paste text → classify with BERT / TPA-BERT → see what words matter"
        "</div>",
        unsafe_allow_html=True,
    )

    # ── Sidebar ──────────────────────────────────────────────
    with st.sidebar:
        st.markdown("### 🤖 Model Selection")

        bert_available = os.path.isdir(BERT_MODEL_PATH)
        tpa_available = os.path.isfile(
            os.path.join(TPA_BERT_MODEL_DIR, TPA_BERT_WEIGHTS)
        )

        if bert_available:
            st.markdown(
                '<span class="status-loaded">✓ BERT model found</span>',
                unsafe_allow_html=True,
            )
        else:
            st.markdown(
                f'<span class="status-missing">✗ BERT not found at {BERT_MODEL_PATH}</span>',
                unsafe_allow_html=True,
            )

        if tpa_available:
            st.markdown(
                '<span class="status-loaded">✓ TPA-BERT model found</span>',
                unsafe_allow_html=True,
            )
        else:
            st.markdown(
                f'<span class="status-missing">✗ TPA-BERT not found</span>',
                unsafe_allow_html=True,
            )

        model_options = []
        if bert_available:
            model_options.append("BERT")
        if tpa_available:
            model_options.append("TPA-BERT")
        if bert_available and tpa_available:
            model_options.append("Both (Compare)")

        if not model_options:
            st.error(
                "No models found! Please check Google Drive is mounted "
                "and model paths are correct."
            )
            st.stop()

        model_choice = st.radio(
            "Choose model(s):",
            model_options,
            index=len(model_options) - 1,
        )

        st.markdown("---")
        st.markdown("### ⚙️ LIME Parameters")

        n_features = st.slider("Number of features", 5, 30, 20)

        show_advanced = st.checkbox("Show advanced settings", value=False)
        if show_advanced:
            n_perturbations = st.slider(
                "Perturbations per sample",
                500,
                5000,
                1000,
                step=250,
                help="More perturbations = more accurate but slower. "
                "1000 is a good balance for interactive use.",
            )
        else:
            n_perturbations = 1000

        st.markdown("---")
        st.markdown("### 📖 How It Works")
        st.markdown(
            "**LIME** (Local Interpretable Model-agnostic Explanations) "
            "perturbs the input text by randomly removing words, observes "
            "how predictions change, and fits a simple linear model to "
            "approximate the classifier locally."
        )
        st.markdown(
            "**Green** words support the prediction. "
            "**Red** words contradict it. "
            "Hover over highlighted words for exact weights."
        )

    # ── Load models ──────────────────────────────────────────
    models = {}

    if model_choice in ("BERT", "Both (Compare)"):
        with st.spinner("Loading BERT model..."):
            bert_model, bert_tokenizer, bert_params = load_bert_model()
            models["BERT"] = {
                "model": bert_model,
                "tokenizer": bert_tokenizer,
                "params": bert_params,
                "is_tpa": False,
            }

    if model_choice in ("TPA-BERT", "Both (Compare)"):
        with st.spinner("Loading TPA-BERT model..."):
            tpa_model, tpa_tokenizer, tpa_params = load_tpa_bert_model()
            models["TPA-BERT"] = {
                "model": tpa_model,
                "tokenizer": tpa_tokenizer,
                "params": tpa_params,
                "is_tpa": True,
            }

    # ── Text Input ───────────────────────────────────────────
    st.markdown("")
    user_text = st.text_area(
        "Enter article text to classify",
        height=200,
        placeholder="Paste or type a news article here...\n\n"
        "Example: WASHINGTON (Reuters) - The U.S. Senate voted on Thursday "
        "to advance a bill that would impose new sanctions on Russia...",
    )

    classify_btn = st.button(
        "🔍 Classify & Explain",
        use_container_width=True,
        type="primary",
        disabled=not user_text.strip(),
    )

    if not user_text.strip():
        st.info("Enter some text above and click **Classify & Explain** to get started.")
        return

    if not classify_btn:
        if "last_results" in st.session_state and st.session_state.get("last_text") == user_text:
            _render_cached_results(
                st.session_state["last_results"], models, user_text, n_features
            )
        return

    # ── Run Classification & LIME ────────────────────────────
    results = {}
    explainer = LimeTextExplainer(class_names=CLASS_NAMES, random_state=42)

    if len(models) == 1:
        model_name = list(models.keys())[0]
        info = models[model_name]
        predict_fn = make_predict_fn(info["model"], info["tokenizer"], info["is_tpa"])

        with st.spinner(
            f"Running LIME with {n_perturbations:,} perturbations on {model_name}..."
        ):
            t0 = time.time()
            probs = predict_fn([user_text])[0]
            pred_label = int(np.argmax(probs))
            exp = explainer.explain_instance(
                user_text,
                predict_fn,
                num_features=n_features,
                num_samples=n_perturbations,
                labels=(0, 1),
            )
            elapsed = time.time() - t0

        lime_weights = exp.as_list(label=pred_label)
        results[model_name] = {
            "pred_label": pred_label,
            "probs": probs.tolist(),
            "lime_weights": lime_weights,
            "elapsed": elapsed,
        }

        render_explanation(
            model_name,
            info["params"],
            pred_label,
            probs.tolist(),
            lime_weights,
            user_text,
            n_features,
            elapsed,
        )

    else:
        for model_name, info in models.items():
            predict_fn = make_predict_fn(
                info["model"], info["tokenizer"], info["is_tpa"]
            )
            with st.spinner(
                f"Running LIME on {model_name} ({n_perturbations:,} perturbations)..."
            ):
                t0 = time.time()
                probs = predict_fn([user_text])[0]
                pred_label = int(np.argmax(probs))
                exp = explainer.explain_instance(
                    user_text,
                    predict_fn,
                    num_features=n_features,
                    num_samples=n_perturbations,
                    labels=(0, 1),
                )
                elapsed = time.time() - t0

            results[model_name] = {
                "pred_label": pred_label,
                "probs": probs.tolist(),
                "lime_weights": exp.as_list(label=pred_label),
                "elapsed": elapsed,
            }

        col_left, col_right = st.columns(2)
        model_names = list(results.keys())

        with col_left:
            name = model_names[0]
            r = results[name]
            render_explanation(
                name,
                models[name]["params"],
                r["pred_label"],
                r["probs"],
                r["lime_weights"],
                user_text,
                n_features,
                r["elapsed"],
            )

        with col_right:
            name = model_names[1]
            r = results[name]
            render_explanation(
                name,
                models[name]["params"],
                r["pred_label"],
                r["probs"],
                r["lime_weights"],
                user_text,
                n_features,
                r["elapsed"],
            )

    st.session_state["last_results"] = results
    st.session_state["last_text"] = user_text

    st.markdown("")
    st.markdown(
        "---\n"
        "<div style='text-align:center; color: rgba(255,255,255,0.25); font-size: 0.75rem;'>"
        "Interactive LIME Explainer · Hover over highlighted words for exact weight values"
        "</div>",
        unsafe_allow_html=True,
    )


def _render_cached_results(results, models, text, n_features):
    """Re-render previously computed results without re-running LIME."""
    if len(results) == 1:
        name = list(results.keys())[0]
        r = results[name]
        if name in models:
            render_explanation(
                name,
                models[name]["params"],
                r["pred_label"],
                r["probs"],
                r["lime_weights"],
                text,
                n_features,
                r["elapsed"],
            )
    else:
        col_left, col_right = st.columns(2)
        model_names = list(results.keys())

        with col_left:
            name = model_names[0]
            r = results[name]
            if name in models:
                render_explanation(
                    name,
                    models[name]["params"],
                    r["pred_label"],
                    r["probs"],
                    r["lime_weights"],
                    text,
                    n_features,
                    r["elapsed"],
                )

        with col_right:
            name = model_names[1]
            r = results[name]
            if name in models:
                render_explanation(
                    name,
                    models[name]["params"],
                    r["pred_label"],
                    r["probs"],
                    r["lime_weights"],
                    text,
                    n_features,
                    r["elapsed"],
                )

    st.markdown("")
    st.markdown(
        "---\n"
        "<div style='text-align:center; color: rgba(255,255,255,0.25); font-size: 0.75rem;'>"
        "Interactive LIME Explainer · Hover over highlighted words for exact weight values"
        "</div>",
        unsafe_allow_html=True,
    )


if __name__ == "__main__":
    main()

Writing /content/app.py


## Cell 3 — Launch Streamlit via ngrok

**First-time setup (one time only):**
1. Sign up at [ngrok.com](https://ngrok.com) and copy your auth token from the [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
2. In Colab, click the **🔑 key icon** in the left sidebar to open **Secrets**
3. Click **+ Add new secret**, set the name to `NGROK_AUTH_TOKEN`, and paste your token as the value
4. Toggle **Notebook access** to ✅ ON

Your token is stored securely in Colab's encrypted vault — it never appears in the notebook.

After running this cell, click the printed URL to open the Streamlit app in a new tab.

In [3]:
import subprocess
import time

# ── Retrieve ngrok token from Colab Secrets ──────────────────
# Set this in the 🔑 Secrets panel (left sidebar) as: NGROK_AUTH_TOKEN
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
except (ImportError, userdata.SecretNotFoundError):
    NGROK_AUTH_TOKEN = None

if not NGROK_AUTH_TOKEN:
    print('⚠ NGROK_AUTH_TOKEN secret not found!')
    print('  1. Click the 🔑 key icon in the left sidebar')
    print('  2. Add a secret named: NGROK_AUTH_TOKEN')
    print('  3. Paste your token from: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('  4. Toggle Notebook access ON')
    raise ValueError('ngrok auth token not found in Colab Secrets')

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✓ ngrok auth token loaded from Colab Secrets')

# Kill any existing Streamlit processes
!pkill -f streamlit 2>/dev/null || true

# Start Streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give Streamlit a moment to start
time.sleep(3)

# Open ngrok tunnel
public_url = ngrok.connect(8501)

print("="*60)
print("🧪 Interactive LIME Explainer is LIVE!")
print("="*60)
print(f"\n🔗 Open this URL in your browser:\n")
print(f"   {public_url}")
print(f"\n   (First load may take ~30s while models load)")
print(f"\n{'='*60}")
print("To stop: Runtime → Interrupt execution, or run Cell 4")

TimeoutException: Requesting secret NGROK_AUTH_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.

## Cell 4 — Teardown (Optional)

Run this cell to stop the Streamlit server and close the ngrok tunnel.

In [ ]:
# Stop everything
from pyngrok import ngrok
ngrok.kill()
!pkill -f streamlit 2>/dev/null || true
print("✓ Streamlit server stopped.")
print("✓ ngrok tunnel closed.")